In [ ]:
def Arjun_recog_approach():
    import time
    import numpy as np
    import cv2
    from IPython.display import display, Image,clear_output

    got.load_models(["face_recognition"])
    got.open_camera()
    print("The camera is on.")

    while True:
        picture_raw = got.read_camera_data()
        if picture_raw is None:
            continue
        
        picture_1D = np.frombuffer(picture_raw,np.uint8)
        picture_bgr = cv2.imdecode(picture_1D,cv2.IMREAD_COLOR)

        face_info = got.get_face_recognition_total_info()
        customer =  None
        
        if face_info and face_info[0][0] == "Arjun":
            customer = face_info[0]

        if customer:
            name,face_x,face_y,face_height,face_width,face_area = customer
            face_x      =int(face_x)
            face_y      =int(face_y)
            face_height =int(face_height)
            face_width  =int(face_width)
            face_area   =int(face_area)

            cv2.rectangle(picture_bgr,(face_x-face_width//2,face_y-face_height//2),(face_x+face_width//2,face_y+face_height//2),(0,255,0),2)
            cv2.putText(picture_bgr,f"{name},face_x:{face_x},face_width:{face_width}",(face_x-face_width//2,face_y-face_height//2-8),cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0),2)

        ok,picture_jpg = cv2.imencode(".jpg",picture_bgr)
        clear_output(wait = True)
        display(Image(picture_jpg.tobytes()))
        
        if not customer:
            got.mecanum_move_xyz(0,10,0)
        elif customer[4]<70:    # face width
            got.mecanum_move_xyz(0,5,0)
        elif customer[1]>340:   # face_x
            got.mecanum_move_xyz(10,5,0)
        elif customer[1]<300:    
            got.mecanum_move_xyz(-10,5,0)
        else:
            got.mecanum_stop()
            break

In [ ]:
Arjun_recog_approach()

In [ ]:
def red_cube_centralizing_approaching_Kp():
    # import libraries
    import time
    import numpy as np
    import cv2
    from IPython.display import display, Image

    # create color range
    lower_color = np.array([0,190,160])
    upper_color = np.array([40,255,240])

    # Kp values
    Kp_x = 0.08
    Kp_y = 0.3

    frame_center = 320
    target_w = 160
    dead_zone_x = 20
    dead_zone_y = 5

    # function 1: get the bgr image
    def picture_bgr():
        picture_raw = got.read_camera_data()
        if picture_raw is None:
            return None
        return cv2.imdecode(np.frombuffer(picture_raw,np.uint8),cv2.IMREAD_COLOR)  

    # function 2: find ball and return bounding box
    def find_ball(frame):
        mask = cv2.inRange(cv2.cvtColor(frame, cv2.COLOR_BGR2HSV),lower_color,upper_color)
        contours,_ = cv2.findContours(mask, cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            return None
        return cv2.boundingRect(max(contours,key=cv2.contourArea))

    # function 3: Kp move and bounding box
    def move_and_draw(frame,x,y,w,h):
        center_x = x + w//2

        # calculate errors
        error_x = center_x - frame_center
        error_y = target_w - w

        #check: close enough and stop
        if abs(error_x) < dead_zone_x and abs(error_y) < dead_zone_y:
            got.mecanum_stop()
            return True
        
        # Kp output
        speed_x = int(Kp_x * error_x)
        speed_y = int(Kp_y * error_y)

        # clamp to safe speed range
        speed_x = max(min(speed_x,25),-25)
        speed_y = max(min(speed_y,25),-25)

        got.mecanum_move_xyz(speed_x, speed_y,0)

        # draw bounding box and debug info  
        cv2.rectangle(frame,(x,y),(x+w,y+h),(0,255,0),2)
        cv2.putText(frame,f"cx:{center_x} ex:{error_x} ey:{error_y},w:{w}",(x,y-10),cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0),2)
        cv2.putText(frame,f"sx:{speed_x} sy:{speed_y}",(x,y-30),cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0),2)
        cv2.circle(frame,(center_x,y+h//2),5,(255,0,0),-1)

    # main program
    got.open_camera()
    ok,picture_jpg = cv2.imencode(".jpg",picture_bgr())
    handle = display(Image(picture_jpg.tobytes()),display_id = True)

    while True:
        frame = picture_bgr()
        if frame is None:
            continue

        result = find_ball(frame)
        if result:
            if move_and_draw(frame,*result):  # close enough and stop
                break
        else:
            got.mecanum_stop()
        
        ok,picture_jpg = cv2.imencode(".jpg",frame)
        if ok:
            handle.update(Image(picture_jpg.tobytes()))
        
        time.sleep(0.05)

In [ ]:
def blue_cube_centralizing_approaching_Kp():
    # import libraries
    import time
    import numpy as np
    import cv2
    from IPython.display import display, Image

    # create color range
    lower_color = np.array([60,210,100])
    upper_color = np.array([140,255,180])

    # Kp values
    Kp_x = 0.08
    Kp_y = 0.3

    frame_center = 320
    target_w = 160
    dead_zone_x = 20
    dead_zone_y = 5

    # function 1: get the bgr image
    def picture_bgr():
        picture_raw = got.read_camera_data()
        if picture_raw is None:
            return None
        return cv2.imdecode(np.frombuffer(picture_raw,np.uint8),cv2.IMREAD_COLOR)  

    # function 2: find ball and return bounding box
    def find_ball(frame):
        mask = cv2.inRange(cv2.cvtColor(frame, cv2.COLOR_BGR2HSV),lower_color,upper_color)
        contours,_ = cv2.findContours(mask, cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            return None
        return cv2.boundingRect(max(contours,key=cv2.contourArea))

    # function 3: Kp move and bounding box
    def move_and_draw(frame,x,y,w,h):
        center_x = x + w//2

        # calculate errors
        error_x = center_x - frame_center
        error_y = target_w - w

        #check: close enough and stop
        if abs(error_x) < dead_zone_x and abs(error_y) < dead_zone_y:
            got.mecanum_stop()
            return True
        
        # Kp output
        speed_x = int(Kp_x * error_x)
        speed_y = int(Kp_y * error_y)

        # clamp to safe speed range
        speed_x = max(min(speed_x,25),-25)
        speed_y = max(min(speed_y,25),-25)

        got.mecanum_move_xyz(speed_x, speed_y,0)

        # draw bounding box and debug info  
        cv2.rectangle(frame,(x,y),(x+w,y+h),(0,255,0),2)
        cv2.putText(frame,f"cx:{center_x} ex:{error_x} ey:{error_y},w:{w}",(x,y-10),cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0),2)
        cv2.putText(frame,f"sx:{speed_x} sy:{speed_y}",(x,y-30),cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0),2)
        cv2.circle(frame,(center_x,y+h//2),5,(255,0,0),-1)

    # main program
    got.open_camera()
    ok,picture_jpg = cv2.imencode(".jpg",picture_bgr())
    handle = display(Image(picture_jpg.tobytes()),display_id = True)

    while True:
        frame = picture_bgr()
        if frame is None:
            continue

        result = find_ball(frame)
        if result:
            if move_and_draw(frame,*result):  # close enough and stop
                break
        else:
            got.mecanum_stop()
        
        ok,picture_jpg = cv2.imencode(".jpg",frame)
        if ok:
            handle.update(Image(picture_jpg.tobytes()))
        
        time.sleep(0.05)